# Brain Tumor 3D Segmentation with MONAI

Import required libraries and packages

In [1]:
import os, shutil, tempfile, time, random, gc, warnings, glob
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'        # Fixes a warning from PyTorch
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional, Literal
from dataclasses import dataclass
from pathlib import Path

from monai.data import Dataset, CacheDataset, DataLoader
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ScaleIntensityd,
    EnsureTyped,
    ToTensord
)

from tqdm import tqdm

# Make plots have guidelines
plt.style.use('ggplot')

# Squash Python warnings
warnings.filterwarnings('ignore')

# Enable Python's Garbage Collector
gc.collect()

2026-02-25 17:56:46.154505434 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
<frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-02-25 17:56:52.823854: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


752

In [2]:
# Directory Structure
ROOT_DIR  = '../BraTS'
TRAIN_DIR = 'BraTS2021_TrainingSet/UPENN-GBM'
VAL_DIR   = 'BraTS2021_ValidationSet/UPENN-GBM'

# Modalities and seg name
MODALITIES = ['t1', 't1ce', 't2', 'flair']

# Define random seed value
SEED = 42

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
#set_determinism(SEED)

# When running on CuDNN backend, it is recommended to set these two options
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision('medium')

In [4]:
@dataclass
class SubjectDict:
    image: List[str]
    label: Optional[str] = None

In [5]:
def get_subjects(root: str, split: Literal['train', 'val']) -> List[SubjectDict]:
    '''
    Walk `root/<split>` and return a list of subject dicts.
    - Training: includes `'label'` key
    - Validation: omits `'label'` key entirely
    '''
    split_path = Path(root) / split
    subjects: List[SubjectDict] = []

    patient_dirs = sorted([d for d in split_path.glob('*') if d.is_dir()])

    for pdir in patient_dirs:
        patient_id = pdir.name
        image_paths = [str(pdir / f'{patient_id}_{mod}.nii.gz') for mod in MODALITIES]

        # Ensure all modalities exist
        missing = [p for p in image_paths if not Path(p).exists()]
        if missing:
            raise FileNotFoundError(
                f'Missing modality files for {patient_id}: {missing}'
            )

        subject = SubjectDict(image=image_paths)

        if split == 'train':
            seg_path = pdir / f'{patient_id}_seg.nii.gz'
            if not seg_path.exists():
                raise FileNotFoundError(f'Missing segmentation for {patient_id}')
            subject.label = str(seg_path)

        subjects.append(subject)

    return subjects

In [6]:
def get_transforms(
    stage: Literal['train', 'val'],
    pixdim: Tuple[float, float, float] = (1.0, 1.0, 1.0)) -> Compose:
    image_keys = ['image']
    label_keys = ['label']

    # Determine which keys to load
    load_keys = image_keys + (label_keys if stage == 'train' else [])
    
    # 🔑 Fixed: Dynamic mode list based on key type
    image_keys_set = {'image'}
    mode_list = [
        'bilinear' if k in image_keys_set else 'nearest'
        for k in load_keys
    ]

    base_transforms = [
        LoadImaged(keys=load_keys),
        EnsureChannelFirstd(keys=load_keys),
        Spacingd(
            keys=load_keys,
            pixdim=pixdim,
            mode=mode_list,
        ),
        Orientationd(keys=load_keys, axcodes='RAS'),
        ScaleIntensityd(keys=image_keys),  # Only scale image, not label
        EnsureTyped(
            keys=load_keys,
            dtype={
                'image': torch.float32,
                'label': torch.long,
            } if 'label' in load_keys else {'image': torch.float32},
        ),
        ToTensord(keys=load_keys),
    ]

    if stage == 'train':
        from monai.transforms import (
            RandFlipd,
            RandRotate90d,
            RandAffined,
            RandCropByPosNegLabeld,
        )
        augmentations = [
            RandFlipd(keys=load_keys, prob=0.5, spatial_axis=0),
            RandRotate90d(keys=load_keys, prob=0.5),
            RandAffined(
                keys=load_keys,
                prob=0.5,
                translate_range=(10, 10, 10),
                rotate_range=(0.1, 0.1, 0.1),
            ),
            RandCropByPosNegLabeld(
                keys=load_keys,
                label_key='label',
                spatial_size=(128, 128, 128),
                pos=1,
                neg=1,
                num_samples=2,
            ),
        ]
        transforms = base_transforms + augmentations
    else:
        transforms = base_transforms

    return Compose(transforms)

In [7]:
def build_dataloaders(
    root: str,
    batch_size: int = 2,
    num_workers: int = 4,
    pin_memory: bool = True,
    cache_rate: float = 1.0) -> Tuple[DataLoader, DataLoader]:
    '''
    Build train and val dataloaders.
    '''
    train_subjects = get_subjects(root, split=TRAIN_DIR)
    val_subjects = get_subjects(root, split=VAL_DIR)

    train_transforms = get_transforms(stage='train')
    val_transforms = get_transforms(stage='val')

    # Use CacheDataset by default; fallback to Dataset if cache_rate=0.0
    train_dataset = CacheDataset(
        data=train_subjects,
        transform=train_transforms,
        cache_rate=cache_rate,
        num_workers=num_workers,
    )

    val_dataset = Dataset(
        data=val_subjects,
        transform=val_transforms,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return train_loader, val_loader

In [8]:
root = Path(ROOT_DIR)
train_loader, val_loader = build_dataloaders(str(root), batch_size=4)
for batch in train_loader:
    images = batch['image']
    labels = batch['label']

Loading dataset:   0%|                                         | 0/403 [00:00<?, ?it/s]


RuntimeError: applying transform <monai.transforms.io.dictionary.LoadImaged object at 0x7f31ea2d4590>